In [0]:

USE main_dev;

-- Enforce Task 2 Idempotency Rules: 
-- CREATE OR REPLACE TABLE ensures that running this file multiple times 
-- completely overwrites the destination target without appending duplicate records.
CREATE OR REPLACE TABLE main_dev.daily_account_summary AS

WITH filtered_transactions AS (
    SELECT 
        account_id,
        -- Truncate string timestamp to a clean UTC Date object
        CAST(TO_TIMESTAMP(transaction_date) AS DATE) AS transaction_date,
        -- Cast string amount to Decimal for precise financial calculations
        CAST(amount AS DECIMAL(18, 4)) AS amount_numeric,
        transaction_type,
        merchant_name,
        merchant_category,
        currency
    FROM main_dev.bronze_transactions
    -- Filter out quarantined data implicitly and keep only completed records
    WHERE status = 'completed'
),

daily_base_metrics AS (
    SELECT
        account_id,
        transaction_date,
        COALESCE(SUM(CASE WHEN transaction_type = 'debit' THEN amount_numeric ELSE 0 END), 0) AS total_debit_amount,
        COALESCE(SUM(CASE WHEN transaction_type = 'credit' THEN amount_numeric ELSE 0 END), 0) AS total_credit_amount,
        COUNT(1) AS transaction_count,
        COUNT(DISTINCT merchant_name) AS distinct_merchants,
        -- Collect distinct currencies into an alphabetized comma-separated string
        ARRAY_JOIN(ARRAY_SORT(ARRAY_DISTINCT(COLLECT_LIST(currency))), ', ') AS currencies
    FROM filtered_transactions
    GROUP BY account_id, transaction_date
),

category_spend_ranking AS (
    SELECT 
        account_id,
        transaction_date,
        merchant_category,
        ROW_NUMBER() OVER (
            PARTITION BY account_id, transaction_date 
            ORDER BY SUM(amount_numeric) DESC, merchant_category ASC
        ) AS category_rank
    FROM filtered_transactions
    GROUP BY account_id, transaction_date, merchant_category
),

top_category_filtered AS (
    SELECT 
        account_id,
        transaction_date,
        merchant_category AS top_category
    FROM category_spend_ranking
    WHERE category_rank = 1
)

SELECT
    m.account_id,
    m.transaction_date,
    CAST(m.total_debit_amount AS DECIMAL(18, 2)) AS total_debit_amount,
    CAST(m.total_credit_amount AS DECIMAL(18, 2)) AS total_credit_amount,
    CAST((m.total_credit_amount - m.total_debit_amount) AS DECIMAL(18, 2)) AS net_amount,
    m.transaction_count,
    m.distinct_merchants,
    t.top_category,
    m.currencies,
    CURRENT_TIMESTAMP() AS updated_at
FROM daily_base_metrics m
LEFT JOIN top_category_filtered t 
    ON m.account_id = t.account_id 
    AND m.transaction_date = t.transaction_date;


In [0]:
%python
import os

# 1. Define the final repository absolute path explicitly
TARGET_DIR = "/Workspace/Repos/souravdw91-star/senior-de-assignment/outputs"
TARGET_FILE = f"{TARGET_DIR}/daily_account_summary_sample.csv"

# 2. Extract dataset cleanly to a Pandas dataframe using Spark driver memory
# Collecting to driver memory isolates the operation completely from DBFS clustering paths
print("Extracting data metrics directly from SQL warehouse matrix...")
sample_df = spark.sql("SELECT * FROM main_dev.daily_account_summary LIMIT 10").toPandas()

# 3. Create a truly local file on the ephemeral Linux driver disk space
# This is isolated on the machine node itself, meaning DBFS rules do not apply
temp_local_path = "/tmp/local_driver_summary_dump.csv"
sample_df.to_csv(temp_local_path, index=False)
print(f"Staged temporary data matrix file cleanly on cluster node disk: {temp_local_path}")

# 4. Generate the target directory path using an absolute bash shell command execution string
# This forces the Databricks system container to map permissions accurately
print("Initializing repository target folder structure...")
os.system(f"mkdir -p {TARGET_DIR}")

# 5. Move the staged file directly into your tracked Git folder location using system cp
print("Flushing sample metrics out to repository directory tracking space...")
exit_code = os.system(f"cp {temp_local_path} {TARGET_FILE}")

# 6. Delete the temporary staged file from cluster storage to keep things tidy
if os.path.exists(temp_local_path):
    os.remove(temp_local_path)

if exit_code == 0:
    print(f"SUCCESS: Successfully generated daily_account_summary_sample.csv inside your outputs folder!")
else:
    print("CRITICAL: File movement command failed. Inspect directory path variables.")
